# Phase 7 — Pricing Intelligence

#### Objective & Scope

This notebook takes active SOA products, retrieves competitor prices, validates matches, compares prices, applies margin rules, and generates pricing recommendations.

In [5]:
## Load libraries
import numpy as np
import pandas as pd
from pathlib import Path

###  Step 1 - Load Competitor Matcher Output

In [6]:

OUTPUT_PATH = Path(
    "D:/STUDY/Data_Science_Courses/PROJECTS/12.Smart_AI-Retail_System/Smart_AI_Retail_System/notebooks/Phase_7_Competitor Price-Matcher/Automation/price_comparison.xlsx"
)

price_comparison_df = pd.read_excel(OUTPUT_PATH)

print("Shape:", price_comparison_df.shape)
display(price_comparison_df.head())

Shape: (11, 7)


,Model,Harvey Norman,Currys,Argos,Donaghy Bros,DSE,Unnamed: 6
0,ES601UK,549.0,549.99,549.99,416,549.00,NaN
1,ES601UKBK,549.0,549.99,549.99,518,549.99,NaN
2,AF400UK,229.0,NOT_FOUND,230,228,229.00,NaN
3,SES876DBL4GUK1,599.0,NOT_FOUND,NOT_FOUND,606.52,599.99,NaN
4,MC1001UK,119.0,119.99,119.99,116.46,119.99,NaN


### Step 2- Validate and clean the generated competitor-price output

In [7]:
# Remove unwanted blank Excel columns
price_comparison_df = price_comparison_df.loc[
    :, ~price_comparison_df.columns.str.contains("^Unnamed")
]

print("Shape after cleanup:", price_comparison_df.shape)
display(price_comparison_df.head())

Shape after cleanup: (11, 6)


,Model,Harvey Norman,Currys,Argos,Donaghy Bros,DSE
0,ES601UK,549.0,549.99,549.99,416,549.00
1,ES601UKBK,549.0,549.99,549.99,518,549.99
2,AF400UK,229.0,NOT_FOUND,230,228,229.00
3,SES876DBL4GUK1,599.0,NOT_FOUND,NOT_FOUND,606.52,599.99
4,MC1001UK,119.0,119.99,119.99,116.46,119.99


In [8]:
competitor_cols = [
    "Harvey Norman",
    "Currys",
    "Argos",
    "Donaghy Bros"
]

price_cols = competitor_cols + ["DSE"]

for col in competitor_cols:
    price_comparison_df[col] = (
        price_comparison_df[col]
        .replace("NOT_FOUND", np.nan)
    )

for col in price_cols:
    price_comparison_df[col] = pd.to_numeric(
        price_comparison_df[col],
        errors="coerce"
    )

C:\Users\singh\AppData\Local\Temp\ipykernel_30368\3321427409.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace("NOT_FOUND", np.nan)


In [9]:
price_comparison_df["Competitor_Count"] = (
    price_comparison_df[competitor_cols]
    .notna()
    .sum(axis=1)
)

In [10]:
display(price_comparison_df.head())

,Model,Harvey Norman,Currys,Argos,Donaghy Bros,DSE,Competitor_Count
0,ES601UK,549.0,549.99,549.99,416.00,549.00,4
1,ES601UKBK,549.0,549.99,549.99,518.00,549.99,4
2,AF400UK,229.0,NaN,230.00,228.00,229.00,3
3,SES876DBL4GUK1,599.0,NaN,NaN,606.52,599.99,2
4,MC1001UK,119.0,119.99,119.99,116.46,119.99,4


### Step 3 — Cheapest Competitor + DSE Price Variance

In [11]:
# Cheapest competitor price
price_comparison_df["Min_Competitor_Price"] = (
    price_comparison_df[competitor_cols]
    .min(axis=1)
)

# Competitor offering the cheapest price
price_comparison_df["Min_Competitor_Name"] = (
    price_comparison_df[competitor_cols]
    .idxmin(axis=1)
)

C:\Users\singh\AppData\Local\Temp\ipykernel_30368\4128583314.py:10: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  .idxmin(axis=1)


In [12]:
price_comparison_df["Price_Difference"] = (
    price_comparison_df["DSE"]
    - price_comparison_df["Min_Competitor_Price"]
)

price_comparison_df["Price_Variance_Pct"] = (
    price_comparison_df["Price_Difference"]
    / price_comparison_df["DSE"]
) * 100

In [13]:
price_comparison_df["Price_Difference"] = (
    price_comparison_df["Price_Difference"].round(2)
)

price_comparison_df["Price_Variance_Pct"] = (
    price_comparison_df["Price_Variance_Pct"].round(2)
)

In [16]:
price_comparison_df = (
    price_comparison_df
    .dropna(subset=["Model"])
    .reset_index(drop=True)
)

print("Clean shape:", price_comparison_df.shape)

display(
    price_comparison_df[
        [
            "Model",
            "DSE",
            "Min_Competitor_Name",
            "Min_Competitor_Price",
            "Price_Difference",
            "Price_Variance_Pct",
            "Competitor_Count"
        ]
    ]
)

Clean shape: (6, 11)


,Model,DSE,Min_Competitor_Name,Min_Competitor_Price,Price_Difference,Price_Variance_Pct,Competitor_Count
0,ES601UK,549.00,Donaghy Bros,416.00,133.00,24.23,4
1,ES601UKBK,549.99,Donaghy Bros,518.00,31.99,5.82,4
2,AF400UK,229.00,Donaghy Bros,228.00,1.00,0.44,3
3,SES876DBL4GUK1,599.99,Harvey Norman,599.00,0.99,0.17,2
4,MC1001UK,119.99,Donaghy Bros,116.46,3.53,2.94,4
5,JBLLIVE770NCBLK,159.00,Currys,89.00,70.00,44.03,2


### Step 4 -  Market Price Position

In [17]:
def classify_market_position(row, tolerance=1.0):

    if pd.isna(row["Min_Competitor_Price"]):
        return "NO_COMPETITOR_FOUND"

    variance = row["Price_Variance_Pct"]

    if variance > tolerance:
        return "DSE_MORE_EXPENSIVE"

    elif variance < -tolerance:
        return "DSE_CHEAPER"

    else:
        return "PRICE_MATCHED"


price_comparison_df["Market_Position"] = (
    price_comparison_df.apply(
        classify_market_position,
        axis=1
    )
)

In [18]:
display(
    price_comparison_df[
        [
            "Model",
            "DSE",
            "Min_Competitor_Name",
            "Min_Competitor_Price",
            "Price_Variance_Pct",
            "Market_Position"
        ]
    ]
)

,Model,DSE,Min_Competitor_Name,Min_Competitor_Price,Price_Variance_Pct,Market_Position
0,ES601UK,549.00,Donaghy Bros,416.00,24.23,DSE_MORE_EXPENSIVE
1,ES601UKBK,549.99,Donaghy Bros,518.00,5.82,DSE_MORE_EXPENSIVE
2,AF400UK,229.00,Donaghy Bros,228.00,0.44,PRICE_MATCHED
3,SES876DBL4GUK1,599.99,Harvey Norman,599.00,0.17,PRICE_MATCHED
4,MC1001UK,119.99,Donaghy Bros,116.46,2.94,DSE_MORE_EXPENSIVE
5,JBLLIVE770NCBLK,159.00,Currys,89.00,44.03,DSE_MORE_EXPENSIVE


### Step 5 — Add Unit Cost and Margin Protection

In [19]:
## Load governed datasets

PROJECT_ROOT = Path.cwd().parent.parent
SALES_PATH = PROJECT_ROOT / "data" / "processed" / "fact_sales.csv"
sales_df = pd.read_csv(SALES_PATH )

print(sales_df.columns.tolist())

['Sales_Record_ID', 'Product_ID', 'Product_Key', 'Source_Month', 'Category', 'Stock Code', 'Description', 'Record_Type', 'Level', 'Sold Period', 'Transaction_Status', 'Unit Cost', 'Unit Price', 'Cost Sales', 'Sales Value', 'Profit', 'Profit %', 'Cost_Sales_Reconciliation_Flag']


In [20]:
cost_lookup = (
    sales_df[
        ["Stock Code", "Unit Cost"]
    ]
    .dropna()
    .drop_duplicates(subset=["Stock Code"])
    .rename(
        columns={
            "Stock Code": "Model",
            "Unit Cost": "Unit_Cost"
        }
    )
)

display(cost_lookup.head())

,Model,Unit_Cost
0,TLS169BOXE,9.31
1,112.204,1.28
2,DLSC500,7.77
3,AF01,2.30
4,SES007NEU0,9.39


In [21]:
price_comparison_df = price_comparison_df.merge(
    cost_lookup,
    on="Model",
    how="left"
)

In [22]:
display(
    price_comparison_df[
        [
            "Model",
            "Unit_Cost",
            "DSE",
            "Min_Competitor_Price"
        ]
    ]
)

,Model,Unit_Cost,DSE,Min_Competitor_Price
0,ES601UK,368.39,549.00,416.00
1,ES601UKBK,386.28,549.99,518.00
2,AF400UK,157.16,229.00,228.00
3,SES876DBL4GUK1,NaN,599.99,599.00
4,MC1001UK,101.99,119.99,116.46
5,JBLLIVE770NCBLK,NaN,159.00,89.00


In [23]:
price_comparison_df["Current_Margin"] = (
    price_comparison_df["DSE"]
    - price_comparison_df["Unit_Cost"]
)

price_comparison_df["Current_Margin_Pct"] = (
    price_comparison_df["Current_Margin"]
    / price_comparison_df["DSE"]
) * 100

In [24]:
MIN_MARGIN_PCT = 10
price_comparison_df["Margin_Floor_Price"] = (
    price_comparison_df["Unit_Cost"]
    / (1 - MIN_MARGIN_PCT / 100)
).round(2)

In [25]:
price_comparison_df["Can_Match_Competitor"] = (
    price_comparison_df["Min_Competitor_Price"]
    >= price_comparison_df["Margin_Floor_Price"]
)

In [26]:
display(
    price_comparison_df[
        [
            "Model",
            "Unit_Cost",
            "DSE",
            "Current_Margin_Pct",
            "Min_Competitor_Name",
            "Min_Competitor_Price",
            "Margin_Floor_Price",
            "Can_Match_Competitor"
        ]
    ]
)

,Model,Unit_Cost,DSE,Current_Margin_Pct,Min_Competitor_Name,Min_Competitor_Price,Margin_Floor_Price,Can_Match_Competitor
0,ES601UK,368.39,549.00,32.897996,Donaghy Bros,416.00,409.32,True
1,ES601UKBK,386.28,549.99,29.765996,Donaghy Bros,518.00,429.20,True
2,AF400UK,157.16,229.00,31.371179,Donaghy Bros,228.00,174.62,True
3,SES876DBL4GUK1,NaN,599.99,NaN,Harvey Norman,599.00,NaN,False
4,MC1001UK,101.99,119.99,15.001250,Donaghy Bros,116.46,113.32,True
5,JBLLIVE770NCBLK,NaN,159.00,NaN,Currys,89.00,NaN,False


In [27]:
def margin_match_status(row):

    if pd.isna(row["Unit_Cost"]):
        return "COST_MISSING"

    if pd.isna(row["Min_Competitor_Price"]):
        return "NO_COMPETITOR_FOUND"

    if row["Min_Competitor_Price"] >= row["Margin_Floor_Price"]:
        return "SAFE_TO_MATCH"

    return "BELOW_MARGIN_FLOOR"


price_comparison_df["Margin_Status"] = (
    price_comparison_df.apply(
        margin_match_status,
        axis=1
    )
)
price_comparison_df["Current_Margin_Pct"] = (
    price_comparison_df["Current_Margin_Pct"]
    .round(2)
)
display(
    price_comparison_df[
        [
            "Model",
            "Unit_Cost",
            "DSE",
            "Current_Margin_Pct",
            "Min_Competitor_Name",
            "Min_Competitor_Price",
            "Margin_Floor_Price",
            "Margin_Status"
        ]
    ]
)

,Model,Unit_Cost,DSE,Current_Margin_Pct,Min_Competitor_Name,Min_Competitor_Price,Margin_Floor_Price,Margin_Status
0,ES601UK,368.39,549.00,32.90,Donaghy Bros,416.00,409.32,SAFE_TO_MATCH
1,ES601UKBK,386.28,549.99,29.77,Donaghy Bros,518.00,429.20,SAFE_TO_MATCH
2,AF400UK,157.16,229.00,31.37,Donaghy Bros,228.00,174.62,SAFE_TO_MATCH
3,SES876DBL4GUK1,NaN,599.99,NaN,Harvey Norman,599.00,NaN,COST_MISSING
4,MC1001UK,101.99,119.99,15.00,Donaghy Bros,116.46,113.32,SAFE_TO_MATCH
5,JBLLIVE770NCBLK,NaN,159.00,NaN,Currys,89.00,NaN,COST_MISSING


### Step 6 — Add Recommendation Reason + Approval Status

In [30]:
PRICE_MATCH_TOLERANCE = 1.0
MANUAL_REVIEW_THRESHOLD = 10.0

def pricing_recommendation(row):

    # Missing cost -> cannot validate margin safely
    if row["Margin_Status"] == "COST_MISSING":
        return "MANUAL_REVIEW"

    # No competitor price available
    if pd.isna(row["Min_Competitor_Price"]):
        return "NO_COMPETITOR_FOUND"

    # Competitor price would breach margin floor
    if row["Margin_Status"] == "BELOW_MARGIN_FLOOR":
        return "HOLD_AT_MARGIN_FLOOR"

    variance = row["Price_Variance_Pct"]

    # Price already effectively matched
    if abs(variance) <= PRICE_MATCH_TOLERANCE:
        return "NO_ACTION"

    # DSE already cheaper
    if variance < -PRICE_MATCH_TOLERANCE:
        return "NO_ACTION"

    # Large price gap -> human review
    if variance > MANUAL_REVIEW_THRESHOLD:
        return "MANUAL_REVIEW"

    # Moderate gap and safe margin
    if variance > PRICE_MATCH_TOLERANCE:
        return "MATCH_COMPETITOR"

    return "NO_ACTION"


price_comparison_df["Pricing_Recommendation"] = (
    price_comparison_df.apply(
        pricing_recommendation,
        axis=1
    )
)

In [31]:
display(
    price_comparison_df[
        [
            "Model",
            "DSE",
            "Min_Competitor_Price",
            "Price_Variance_Pct",
            "Margin_Status",
            "Pricing_Recommendation"
        ]
    ]
)

,Model,DSE,Min_Competitor_Price,Price_Variance_Pct,Margin_Status,Pricing_Recommendation
0,ES601UK,549.00,416.00,24.23,SAFE_TO_MATCH,MANUAL_REVIEW
1,ES601UKBK,549.99,518.00,5.82,SAFE_TO_MATCH,MATCH_COMPETITOR
2,AF400UK,229.00,228.00,0.44,SAFE_TO_MATCH,NO_ACTION
3,SES876DBL4GUK1,599.99,599.00,0.17,COST_MISSING,MANUAL_REVIEW
4,MC1001UK,119.99,116.46,2.94,SAFE_TO_MATCH,MATCH_COMPETITOR
5,JBLLIVE770NCBLK,159.00,89.00,44.03,COST_MISSING,MANUAL_REVIEW


In [33]:
def proposed_price(row):

    if row["Pricing_Recommendation"] == "MATCH_COMPETITOR":
        return row["Min_Competitor_Price"]

    if row["Pricing_Recommendation"] == "HOLD_AT_MARGIN_FLOOR":
        return row["Margin_Floor_Price"]

    return row["DSE"]


price_comparison_df["Proposed_Price"] = (
    price_comparison_df.apply(
        proposed_price,
        axis=1
    )
)

def recommendation_reason(row):

    if row["Pricing_Recommendation"] == "MANUAL_REVIEW":

        if row["Margin_Status"] == "COST_MISSING":
            return "Unit cost missing; margin safety cannot be validated"

        if row["Price_Variance_Pct"] > MANUAL_REVIEW_THRESHOLD:
            return "Price gap exceeds manual-review threshold"

        return "Manual validation required"

    if row["Pricing_Recommendation"] == "MATCH_COMPETITOR":
        return (
            f"DSE is {row['Price_Variance_Pct']:.2f}% above "
            f"{row['Min_Competitor_Name']} and match remains above margin floor"
        )

    if row["Pricing_Recommendation"] == "HOLD_AT_MARGIN_FLOOR":
        return "Competitor price is below minimum allowed margin floor"

    if row["Pricing_Recommendation"] == "NO_ACTION":
        return "Current DSE price is within acceptable market tolerance"

    if row["Pricing_Recommendation"] == "NO_COMPETITOR_FOUND":
        return "No valid competitor price available"

    return "No reason available"


price_comparison_df["Recommendation_Reason"] = (
    price_comparison_df.apply(
        recommendation_reason,
        axis=1
    )
)

def approval_status(row):

    if row["Pricing_Recommendation"] == "MATCH_COMPETITOR":
        return "AUTO_APPROVED"

    if row["Pricing_Recommendation"] == "MANUAL_REVIEW":
        return "PENDING_MANAGER_REVIEW"

    if row["Pricing_Recommendation"] == "HOLD_AT_MARGIN_FLOOR":
        return "BLOCKED"

    if row["Pricing_Recommendation"] == "NO_ACTION":
        return "NO_APPROVAL_REQUIRED"

    if row["Pricing_Recommendation"] == "NO_COMPETITOR_FOUND":
        return "NO_ACTION"

    return "REVIEW"


price_comparison_df["Approval_Status"] = (
    price_comparison_df.apply(
        approval_status,
        axis=1
    )
)

In [34]:
display(
    price_comparison_df[
        [
            "Model",
            "DSE",
            "Min_Competitor_Name",
            "Min_Competitor_Price",
            "Pricing_Recommendation",
            "Proposed_Price",
            "Approval_Status",
            "Recommendation_Reason"
        ]
    ]
)

,Model,DSE,Min_Competitor_Name,Min_Competitor_Price,Pricing_Recommendation,Proposed_Price,Approval_Status,Recommendation_Reason
0,ES601UK,549.00,Donaghy Bros,416.00,MANUAL_REVIEW,549.00,PENDING_MANAGER_REVIEW,Price gap exceeds manual-review threshold
1,ES601UKBK,549.99,Donaghy Bros,518.00,MATCH_COMPETITOR,518.00,AUTO_APPROVED,DSE is 5.82% above Donaghy Bros and match rema...
2,AF400UK,229.00,Donaghy Bros,228.00,NO_ACTION,229.00,NO_APPROVAL_REQUIRED,Current DSE price is within acceptable market ...
3,SES876DBL4GUK1,599.99,Harvey Norman,599.00,MANUAL_REVIEW,599.99,PENDING_MANAGER_REVIEW,Unit cost missing; margin safety cannot be val...
4,MC1001UK,119.99,Donaghy Bros,116.46,MATCH_COMPETITOR,116.46,AUTO_APPROVED,DSE is 2.94% above Donaghy Bros and match rema...
5,JBLLIVE770NCBLK,159.00,Currys,89.00,MANUAL_REVIEW,159.00,PENDING_MANAGER_REVIEW,Unit cost missing; margin safety cannot be val...


In [35]:
final_pricing_df = price_comparison_df[
    [
        "Model",
        "DSE",
        "Min_Competitor_Name",
        "Min_Competitor_Price",
        "Price_Difference",
        "Price_Variance_Pct",
        "Unit_Cost",
        "Current_Margin_Pct",
        "Margin_Floor_Price",
        "Margin_Status",
        "Pricing_Recommendation",
        "Proposed_Price",
        "Approval_Status",
        "Recommendation_Reason"
    ]
].copy()

display(final_pricing_df)

,Model,DSE,Min_Competitor_Name,Min_Competitor_Price,Price_Difference,Price_Variance_Pct,Unit_Cost,Current_Margin_Pct,Margin_Floor_Price,Margin_Status,Pricing_Recommendation,Proposed_Price,Approval_Status,Recommendation_Reason
0,ES601UK,549.00,Donaghy Bros,416.00,133.00,24.23,368.39,32.90,409.32,SAFE_TO_MATCH,MANUAL_REVIEW,549.00,PENDING_MANAGER_REVIEW,Price gap exceeds manual-review threshold
1,ES601UKBK,549.99,Donaghy Bros,518.00,31.99,5.82,386.28,29.77,429.20,SAFE_TO_MATCH,MATCH_COMPETITOR,518.00,AUTO_APPROVED,DSE is 5.82% above Donaghy Bros and match rema...
2,AF400UK,229.00,Donaghy Bros,228.00,1.00,0.44,157.16,31.37,174.62,SAFE_TO_MATCH,NO_ACTION,229.00,NO_APPROVAL_REQUIRED,Current DSE price is within acceptable market ...
3,SES876DBL4GUK1,599.99,Harvey Norman,599.00,0.99,0.17,NaN,NaN,NaN,COST_MISSING,MANUAL_REVIEW,599.99,PENDING_MANAGER_REVIEW,Unit cost missing; margin safety cannot be val...
4,MC1001UK,119.99,Donaghy Bros,116.46,3.53,2.94,101.99,15.00,113.32,SAFE_TO_MATCH,MATCH_COMPETITOR,116.46,AUTO_APPROVED,DSE is 2.94% above Donaghy Bros and match rema...
5,JBLLIVE770NCBLK,159.00,Currys,89.00,70.00,44.03,NaN,NaN,NaN,COST_MISSING,MANUAL_REVIEW,159.00,PENDING_MANAGER_REVIEW,Unit cost missing; margin safety cannot be val...


In [ ]:
## Save the file

""" final_pricing_df.to_excel(
    "pricing_recommendations.xlsx",
    index=False
) """

###  Step 7 - — Audit Log

In [ ]:
from datetime import datetime

run_timestamp = datetime.now()

audit_log_df = final_pricing_df.copy()

audit_log_df["Decision_Timestamp"] = run_timestamp
audit_log_df["Decision_Source"] = "Pricing_Intelligence_Rule_Engine"

audit_log_df["Approver"] = audit_log_df["Approval_Status"].apply(
    lambda x: "AUTO"
    if x == "AUTO_APPROVED"
    else None
)

In [38]:
audit_log_df["Audit_ID"] = [
    f"PRICE_{run_timestamp.strftime('%Y%m%d_%H%M%S')}_{i+1:03d}"
    for i in range(len(audit_log_df))
]

In [39]:
audit_log_df = audit_log_df[
    [
        "Audit_ID",
        "Decision_Timestamp",
        "Model",
        "DSE",
        "Min_Competitor_Name",
        "Min_Competitor_Price",
        "Price_Variance_Pct",
        "Unit_Cost",
        "Margin_Floor_Price",
        "Margin_Status",
        "Pricing_Recommendation",
        "Proposed_Price",
        "Approval_Status",
        "Approver",
        "Recommendation_Reason",
        "Decision_Source"
    ]
]

In [40]:
display(audit_log_df)

,Audit_ID,Decision_Timestamp,Model,DSE,Min_Competitor_Name,Min_Competitor_Price,Price_Variance_Pct,Unit_Cost,Margin_Floor_Price,Margin_Status,Pricing_Recommendation,Proposed_Price,Approval_Status,Approver,Recommendation_Reason,Decision_Source
0,PRICE_20260912_183015_001,2026-09-12 18:30:15.943846,ES601UK,549.00,Donaghy Bros,416.00,24.23,368.39,409.32,SAFE_TO_MATCH,MANUAL_REVIEW,549.00,PENDING_MANAGER_REVIEW,None,Price gap exceeds manual-review threshold,Pricing_Intelligence_Rule_Engine
1,PRICE_20260912_183015_002,2026-09-12 18:30:15.943846,ES601UKBK,549.99,Donaghy Bros,518.00,5.82,386.28,429.20,SAFE_TO_MATCH,MATCH_COMPETITOR,518.00,AUTO_APPROVED,AUTO,DSE is 5.82% above Donaghy Bros and match rema...,Pricing_Intelligence_Rule_Engine
2,PRICE_20260912_183015_003,2026-09-12 18:30:15.943846,AF400UK,229.00,Donaghy Bros,228.00,0.44,157.16,174.62,SAFE_TO_MATCH,NO_ACTION,229.00,NO_APPROVAL_REQUIRED,None,Current DSE price is within acceptable market ...,Pricing_Intelligence_Rule_Engine
3,PRICE_20260912_183015_004,2026-09-12 18:30:15.943846,SES876DBL4GUK1,599.99,Harvey Norman,599.00,0.17,NaN,NaN,COST_MISSING,MANUAL_REVIEW,599.99,PENDING_MANAGER_REVIEW,None,Unit cost missing; margin safety cannot be val...,Pricing_Intelligence_Rule_Engine
4,PRICE_20260912_183015_005,2026-09-12 18:30:15.943846,MC1001UK,119.99,Donaghy Bros,116.46,2.94,101.99,113.32,SAFE_TO_MATCH,MATCH_COMPETITOR,116.46,AUTO_APPROVED,AUTO,DSE is 2.94% above Donaghy Bros and match rema...,Pricing_Intelligence_Rule_Engine
5,PRICE_20260912_183015_006,2026-09-12 18:30:15.943846,JBLLIVE770NCBLK,159.00,Currys,89.00,44.03,NaN,NaN,COST_MISSING,MANUAL_REVIEW,159.00,PENDING_MANAGER_REVIEW,None,Unit cost missing; margin safety cannot be val...,Pricing_Intelligence_Rule_Engine


In [ ]:
## Save Audit log
""" audit_log_df.to_excel(
    "pricing_audit_log.xlsx",
    index=False
) """

### Step 8 — Final Validation

In [42]:
# 1. Auto-approved recommendations must never breach the margin floor
auto_floor_breaches = audit_log_df[
    (audit_log_df["Approval_Status"] == "AUTO_APPROVED") &
    (audit_log_df["Margin_Status"] == "BELOW_MARGIN_FLOOR")
]

print("Auto-approved margin floor breaches:", len(auto_floor_breaches))

Auto-approved margin floor breaches: 0


In [43]:
missing_cost_wrong_route = audit_log_df[
    (audit_log_df["Margin_Status"] == "COST_MISSING") &
    (audit_log_df["Approval_Status"] != "PENDING_MANAGER_REVIEW")
]

print("Missing-cost routing errors:", len(missing_cost_wrong_route))

Missing-cost routing errors: 0


In [44]:
large_gap_auto_approved = audit_log_df[
    (audit_log_df["Price_Variance_Pct"] > MANUAL_REVIEW_THRESHOLD) &
    (audit_log_df["Approval_Status"] == "AUTO_APPROVED")
]

print("Large-gap auto-approval errors:", len(large_gap_auto_approved))

Large-gap auto-approval errors: 0


In [45]:
invalid_proposed_prices = audit_log_df[
    audit_log_df["Proposed_Price"].isna() |
    (audit_log_df["Proposed_Price"] <= 0)
]

print("Invalid proposed prices:", len(invalid_proposed_prices))

Invalid proposed prices: 0


In [46]:
missing_reasons = audit_log_df[
    audit_log_df["Recommendation_Reason"].isna() |
    (audit_log_df["Recommendation_Reason"].str.strip() == "")
]

print("Missing decision reasons:", len(missing_reasons))

Missing decision reasons: 0


In [47]:
validation_summary = {
    "Auto_Approved_Margin_Breaches": len(auto_floor_breaches),
    "Missing_Cost_Routing_Errors": len(missing_cost_wrong_route),
    "Large_Gap_Auto_Approval_Errors": len(large_gap_auto_approved),
    "Invalid_Proposed_Prices": len(invalid_proposed_prices),
    "Missing_Decision_Reasons": len(missing_reasons),
}

validation_summary

{'Auto_Approved_Margin_Breaches': 0,
 'Missing_Cost_Routing_Errors': 0,
 'Large_Gap_Auto_Approval_Errors': 0,
 'Invalid_Proposed_Prices': 0,
 'Missing_Decision_Reasons': 0}

The Pricing Intelligence workflow was successfully implemented and validated.

The solution now:

- Reads competitor-pricing output generated from active SOA models.
- Identifies the lowest competitor price across multiple retailers.
- Compares competitor pricing against the current DSE selling price.
- Calculates price variance and current margin.
- Applies a configurable minimum-margin floor.
- Routes large pricing gaps and missing-cost cases to manual review.
- Auto-approves only safe competitor matches within defined thresholds.
- Generates a proposed selling price with an explainable recommendation reason.
- Produces an audit log containing the decision, approval status, timestamp and pricing context.
- Validates that no unsafe pricing decisions are silently auto-approved.